# Databricks/PySpark for Data Analysts/Engineers — Module 8: UDFs and Custom Functions

Sometimes no function in `pyspark.sql.functions` does exactly what we need -- that's when
we reach for a **UDF** (*User Defined Function*, a function defined by the user in plain
Python). This module shows how to write them, but above all -- why it's worth AVOIDING
them whenever possible.

## Table of contents
1. [When you really need a UDF (and when you don't)](#sec1)
2. [A plain Python UDF](#sec2)
3. [The performance cost of UDFs](#sec3)
4. [Pandas UDF -- a vectorized alternative](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


<a id="sec1"></a>
## 1. When you really need a UDF (and when you don't)

Before writing a UDF, check whether the same result can be achieved with built-in
functions (`pyspark.sql.functions`, Module 3) -- the set of `when/otherwise`, string,
date, and math functions covers a surprising number of cases. A UDF only makes sense
once the logic is genuinely custom: a complex algorithm, a call into an external Python
library (e.g. `re` for elaborate regular expressions, an NLP library), or logic hard to
express as a single SQL expression.

> ⚠️ **The most common beginner mistake: a UDF where when/otherwise would suffice**
>
> Example: an `@udf` that classifies price into `'Cheap'`/`'Mid'`/`'Expensive'` via Python `if/elif` -- that's EXACTLY what `F.when().when().otherwise()` from Module 3 does faster and without any UDF. Always check the built-in functions first.

<a id="sec2"></a>
## 2. A plain Python UDF

`@F.udf(return_type)` turns a plain Python function into a UDF you can use inside
`withColumn`/`select`. IMPORTANT: you must explicitly give the return type
(`pyspark.sql.types`) -- Spark cannot infer it from Python code on its own.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

@F.udf(StringType())
def shorten_city_name(city):
    if city is None:
        return None
    return city[:3].upper()

customers.withColumn("city_code", shorten_city_name(F.col("city"))).select(
    "customer_id", "city", "city_code"
).show(5)


> ⚡ **Registering a UDF for use in Spark SQL**
>
> If you want to use the same function in a `spark.sql(...)` query (Module 5), register it via `spark.udf.register("sql_name", shorten_city_name)` -- only then is it visible as a plain SQL function in text queries.

<a id="sec3"></a>
## 3. The performance cost of UDFs

Built-in functions (`pyspark.sql.functions`) run INSIDE the JVM -- Catalyst knows their
logic and can optimize them together with the rest of the plan. A plain Python UDF
requires SERIALIZING each row from the JVM to a Python process, running the function
in Python, and DESERIALIZING the result back into the JVM -- for every row separately.
That's a serious cost on large datasets.

In [ ]:
# .explain() on a UDF doesn't show WHAT the function does -- Catalyst sees it as a "black box"
customers.withColumn("city_code", shorten_city_name(F.col("city"))).explain()


> ⚠️ **PythonUDF in the execution plan -- a warning sign**
>
> If you see a `BatchEvalPython` node (or similar) in `.explain()` -- that's a sign Spark has to switch between the JVM and Python for that column. A single UDF on small data is nothing to worry about, but a chain of several UDFs on a large dataset is often the main bottleneck of the whole pipeline -- the first place to check when something runs slower than expected.

<a id="sec4"></a>
## 4. Pandas UDF -- a vectorized alternative

A **pandas UDF** (formerly *Vectorized UDF*) processes data in BATCHES (as a
`pandas.Series`), using **Apache Arrow** for a much more efficient data transfer
between the JVM and Python -- instead of row by row. The syntax is similar, but the
function operates on an entire column at once.

In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(StringType())
def shorten_city_name_pandas(city: pd.Series) -> pd.Series:
    return city.str[:3].str.upper()

# A pandas UDF uses Apache Arrow to transfer data between the JVM and Python -- this
# requires a compatible Java version and the Arrow library bundled with Spark (see the
# note below). On an incompatible Java/Arrow pairing it raises an exception -- the
# try/except below reports that gracefully instead of crashing the whole notebook run.
try:
    customers.withColumn(
        "city_code_pandas", shorten_city_name_pandas(F.col("city"))
    ).select("customer_id", "city", "city_code_pandas").show(5)
except Exception as e:
    print("Pandas UDF doesn't work in this environment (Java/Apache Arrow mismatch).")
    print(f"Details: {type(e).__name__}")


> ⚠️ **A pandas UDF needs a compatible Java and Apache Arrow version**
>
> Pandas UDFs rely on **Apache Arrow** to pass data between the JVM and Python -- this requires the Java version Spark runs on to be compatible with the Arrow library version bundled with that PySpark release. On a very new Java version (e.g. JDK 21) paired with an older bundled Arrow (as in this course), calling a pandas UDF can raise an `UnsupportedOperationException` -- that's an environment limitation, not a bug in the code. In Databricks, Java and Arrow are always paired so they work together, so the exact same code runs there without any changes.

> 🧱 **When a pandas UDF is really worth it**
>
> The performance difference is negligible on small datasets (like ours in this course), but grows significantly on large, distributed data -- especially when the Python logic uses vectorized libraries (`numpy`, `pandas`, ML models). In Databricks, pandas UDFs are the standard tool for integrating custom Python/ML logic into Spark pipelines without giving up performance.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- when a UDF is truly needed -- and that it's always worth checking built-in functions first,
- a plain Python UDF (`@F.udf(return_type)`) -- syntax and the explicit return type,
- the performance cost: row-by-row serialization/deserialization between the JVM and Python,
- pandas UDF (`@pandas_udf`) -- a vectorized alternative built on Apache Arrow.

### 📝 Exercise 1: A UDF computing region length

Write a plain Python UDF returning the string length (`len()`) of the `region` column from `employees` -- think about whether this could be done without a UDF (hint: yes, `F.length()`).

<details>
<summary>Show solution</summary>

```python
from pyspark.sql.types import IntegerType

@F.udf(IntegerType())
def region_length(region):
    return len(region) if region is not None else None

employees.withColumn("length", region_length(F.col("region"))).show()
# Without a UDF: employees.withColumn("length", F.length("region")).show()
```

</details>

### 📝 Exercise 2: A UDF with custom logic

Write a Python UDF that, for the `status` column, returns a label: `'OK'` for `'Delivered'`, `'IN TRANSIT'` for `'Shipped'`, `'CANCELLED'` for `'Cancelled'`, `'PENDING'` otherwise.

<details>
<summary>Show solution</summary>

```python
@F.udf(StringType())
def status_label(status):
    mapping = {"Delivered": "OK", "Shipped": "IN TRANSIT", "Cancelled": "CANCELLED"}
    return mapping.get(status, "PENDING")

orders.withColumn("status_label", status_label(F.col("status"))).select(
    "order_id", "status", "status_label"
).show(10)
```

</details>

### 📝 Exercise 3: The same UDF as a pandas UDF

Rewrite the UDF from exercise 2 as a pandas UDF, operating on the whole column (`pandas.Series`) at once instead of row by row.

<details>
<summary>Show solution</summary>

```python
@pandas_udf(StringType())
def status_label_pandas(status: pd.Series) -> pd.Series:
    mapping = {"Delivered": "OK", "Shipped": "IN TRANSIT", "Cancelled": "CANCELLED"}
    return status.map(mapping).fillna("PENDING")

orders.withColumn("status_label", status_label_pandas(F.col("status"))).select(
    "order_id", "status", "status_label"
).show(10)
```

</details>

### 📝 Exercise 4: Registering a UDF for Spark SQL

Register the `region_length` UDF from exercise 1 under the SQL name `length_sql` and use it in a `spark.sql(...)` query against the `employees_view` view (first register the view via `createOrReplaceTempView`).

<details>
<summary>Show solution</summary>

```python
employees.createOrReplaceTempView("employees_view")
spark.udf.register("length_sql", region_length)

spark.sql("""
    SELECT employee_id, region, length_sql(region) AS length
    FROM employees_view
""").show()
```

</details>

> 🔥 **Challenge: comparing execution plans -- a plain UDF vs. built-in functions**
>
> Categorize `products` into `price_tier` TWO ways: (1) a plain Python UDF with `if/elif`, (2) built-in `when/otherwise` functions (as in Module 3). Call `.explain()` on both results and compare -- which plan shows a `BatchEvalPython` (or similar) node, and which doesn't?

<details>
<summary>Show solution</summary>

```python
@F.udf(StringType())
def price_tier_udf(price):
    if price < 100:
        return "Cheap"
    elif price <= 500:
        return "Mid"
    return "Expensive"

udf_version = products.withColumn("price_tier", price_tier_udf(F.col("price")))
builtin_version = products.withColumn(
    "price_tier",
    F.when(F.col("price") < 100, "Cheap")
    .when(F.col("price") <= 500, "Mid")
    .otherwise("Expensive"),
)

print("=== UDF version ===")
udf_version.explain()
print("=== Built-in functions version ===")
builtin_version.explain()
```

The UDF version's plan should include a node related to Python evaluation (e.g. BatchEvalPython) -- the when/otherwise version doesn't have one, since all the logic stays inside the JVM and is fully optimized by Catalyst.

</details>

## What's next?

In **Module 9** we'll cover optimization and performance: in-memory partitioning,
`cache()`/`persist()`, an overview of Catalyst/Tungsten, and reading `.explain()` plans
in more depth.